### LCEL Deepdive

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
prompt = ChatPromptTemplate.from_template("Explain about {topic} in detail")
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain.invoke({"topic": "ice cream"})

'Ice cream is one of the world\'s most beloved and iconic desserts, enjoyed across cultures and generations. It\'s far more than just "frozen sugar water"; it\'s a complex emulsion and foam, a delightful blend of science, art, and culinary tradition.\n\nLet\'s dive into the world of ice cream in detail:\n\n## What is Ice Cream?\n\nAt its core, ice cream is a **frozen dessert** typically made from a **dairy base** (milk, cream), **sweetened** (sugar, corn syrup), and **flavored**. Its characteristic texture comes from the careful balance of **ice crystals, fat globules, air cells, and unfrozen water**.\n\n## Key Components & Ingredients:\n\n1.  **Dairy (Milk Fat & Milk Solids Non-Fat - MSNF):**\n    *   **Milk Fat (Cream):** This is the primary determinant of ice cream\'s richness and smooth texture. Higher fat content generally leads to a creamier, more luxurious mouthfeel. It coats ice crystals, preventing them from growing large, and contributes to the emulsion.\n    *   **Milk Solid

In [5]:
print(prompt.invoke({"topic": "ice cream"}))

messages=[HumanMessage(content='Explain about ice cream in detail', additional_kwargs={}, response_metadata={})]


In [6]:
from langchain_core.messages.human import HumanMessage

messages = [HumanMessage(content='tell me a short joke about ice cream')]
model.invoke(messages)

AIMessage(content='Why did the ice cream get a ticket?\n\nBecause it was double parked!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--c039f566-57ab-4801-9df8-b0a4c3a76023-0', usage_metadata={'input_tokens': 9, 'output_tokens': 756, 'total_tokens': 765, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 740}})

### What is this "|" in Python?

In [7]:
from abc import ABC, abstractmethod

class CRunnable(ABC):
    def __init__(self):
        self.next = None

    @abstractmethod
    def process(self, data):
        """
        This method must be implemented by subclasses to define
        data processing behavior.
        """
        pass

    def invoke(self, data):
        processed_data = self.process(data)
        if self.next is not None:
            return self.next.invoke(processed_data)
        return processed_data

    def __or__(self, other):
        return CRunnableSequence(self, other)

class CRunnableSequence(CRunnable):
    def __init__(self, first, second):
        super().__init__()
        self.first = first
        self.second = second

    def process(self, data):
        return data

    def invoke(self, data):
        first_result = self.first.invoke(data)
        return self.second.invoke(first_result)



In [8]:
class AddTen(CRunnable):
    def process(self, data):
        print("AddTen: ", data)
        return data + 10

class MultiplyByTwo(CRunnable):
    def process(self, data):
        print("Multiply by 2: ", data)
        return data * 2

class ConvertToString(CRunnable):
    def process(self, data):
        print("Convert to string: ", data)
        return f"Result: {data}"

In [9]:
a = AddTen()
b = MultiplyByTwo()
c = ConvertToString()

chain = a | b | c

In [10]:
result = chain.invoke(10)
print(result)

AddTen:  10
Multiply by 2:  20
Convert to string:  40
Result: 40


### Runnables from LangChain

In [11]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [12]:
chain = RunnablePassthrough() | RunnablePassthrough () | RunnablePassthrough ()
chain.invoke("hello")

'hello'

In [13]:
def input_to_upper(input: str):
    output = input.upper()
    return output

In [14]:
chain = RunnablePassthrough() | RunnableLambda(input_to_upper) | RunnablePassthrough()
chain.invoke("hello")

'HELLO'

In [15]:
chain = RunnableParallel({"x": RunnablePassthrough(), "y": RunnablePassthrough()})

In [16]:
chain.invoke("hello")

{'x': 'hello', 'y': 'hello'}

In [17]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'},
 'y': {'input': 'hello', 'input2': 'goodbye'}}

In [18]:
chain = RunnableParallel({"x": RunnablePassthrough(), "y": lambda z: z["input2"]})

In [19]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': 'goodbye'}

### Nested chains - now it gets more complicated!

In [20]:
def find_keys_to_uppercase(input: dict):
    output = input.get("input", "not found").upper()
    return output

In [21]:
chain = RunnableParallel({"x": RunnablePassthrough() | RunnableLambda(find_keys_to_uppercase), "y": lambda z: z["input2"]})

In [22]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': 'HELLO', 'y': 'goodbye'}

In [23]:
chain = RunnableParallel({"x": RunnablePassthrough()})

def assign_func(input):
    return 100

def multiply(input):
    return input * 10

In [24]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}}

In [25]:
chain = RunnableParallel({"x": RunnablePassthrough()}).assign(extra=RunnableLambda(assign_func))

In [26]:
result = chain.invoke({"input": "hello", "input2": "goodbye"})
print(result)

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'extra': 100}


### Combine multiple chains (incl. coercion)

In [28]:
def extractor(input: dict):
    return input.get("extra", "Key not found")

def cupper(upper: str):
    return str(upper).upper()

new_chain = RunnableLambda(extractor) | RunnableLambda(cupper)

In [29]:
new_chain.invoke({"extra": "test"})

'TEST'

In [30]:
final_chain = chain | new_chain
final_chain.invoke({"input": "hello", "input2": "goodbye"})

'100'

### Real Work example

In [36]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


vectorstore = FAISS.from_texts(
    ["Cats love tuna"], embedding=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template=template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | prompt
    | ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    | StrOutputParser()
)

In [37]:
rag_chain.invoke("What do cats like to eat?")

'Cats like to eat tuna.'

In [38]:
RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()}).invoke("What do cats like to eat?")

{'context': 'Cats love tuna', 'question': 'What do cats like to eat?'}

In [ ]:
prompt.invoke({"context": "Cats love tuna", "question": "What do cats like to eat?"})

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\nCats love thuna\n\nQuestion: What do cats like to eat?\n', additional_kwargs={}, response_metadata={})])

In [42]:
ChatGoogleGenerativeAI(model="gemini-2.5-flash").invoke(prompt.invoke({"context": "Cats love tuna", "question": "What do cats like to eat?"}))

AIMessage(content='Cats like to eat tuna.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--5cdc50cb-e1d5-483c-b1c1-7a2dccee7214-0', usage_metadata={'input_tokens': 26, 'output_tokens': 148, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 142}})